In [1]:
from llama_index.core.node_parser import SentenceSplitter, JSONNodeParser
from llama_index.llms.llama_cpp import LlamaCPP
from llama_index.readers.json import JSONReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import Settings
from llama_index.core import StorageContext
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.postgres import PGVectorStore
from llama_index.core import Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
from transformers import AutoTokenizer
from llama_index.core import set_global_tokenizer
from llama_index.core.node_parser import HTMLNodeParser
from pathlib import Path
from bs4 import BeautifulSoup
import psycopg 
from llama_index.core import PromptTemplate
import os
from dotenv import load_dotenv
import json
from llama_index.core.schema import TextNode, NodeRelationship, RelatedNodeInfo

In [2]:
load_dotenv("/setup/on.env")
pg_user = os.getenv("POSTGRES_USER")
pg_db = os.getenv("POSTGRES_DB")
pg_pwd = os.getenv("POSTGRES_PASSWORD")

In [3]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-14B-Instruct")

Settings.embed_model = HuggingFaceEmbedding(
    model_name = "BAAI/bge-base-en-v1.5"
)

set_global_tokenizer(tokenizer.encode)

In [4]:
data_dir = "/notebooks/data/ttlg-posts"
all_json = []
for ext in ["*.json"]:
    for path in Path(data_dir).rglob(ext):
        with open(path, 'r') as f:
            all_json.append(json.load(f))

In [5]:
nodes = []
for thread in all_json:
    metadata = {
        'title': thread['title'],
        'users': [i['username'] for i in thread['posts']],
        'date': thread['posts'][0]['date']
    }
    node = TextNode(text=json.dumps(thread), metadata=metadata)
    
    nodes.append(node)

In [6]:
# node_parser = SentenceSplitter(chunk_size=1024, chunk_overlap=20)

# nodes_ = node_parser.get_nodes_from_documents(
#     nodes, show_progress=False
# )

In [8]:
len(nodes)

1041

In [9]:
# reader = JSONReader(levels_back=1, collapse_length=50, clean_json=False)
# data_dir = "/notebooks/data/ttlg-posts"
# all_docs = []
# nodes = []
# for ext in ["*.json"]:
#     for path in Path(data_dir).rglob(ext):
#         # all_docs.extend(FlatReader().load_data(path))
#         nodes.extend(node_parser.get_nodes_from_documents(
#             FlatReader().load_data(path), show_progress=True
#         ))
#         # all_docs.extend(reader.load_data(input_file=path, extra_info={}))

In [10]:
# node_parser = JSONNodeParser()

# nodes = node_parser.get_nodes_from_documents(
#     all_docs, show_progress=True
# )

In [11]:
def drop(name):
    with psycopg.connect(
        f"host=postgres dbname={pg_db} user={pg_user} password={pg_pwd}"
    ) as conn:
        with conn.cursor() as cur:
            cur.execute(f"""
                drop table if exists {name};
                """)
            conn.commit()


drop("data_json")

In [12]:
vector_store = PGVectorStore.from_params(
    database=pg_db,
    host="postgres",
    password=pg_pwd,
    port=5432,
    user=pg_user,
    table_name="json",
    embed_dim=768,
    hnsw_kwargs={
        "hnsw_m": 14,
        "hnsw_ef_construction": 72,
        "hnsw_ef_search": 52,
        "hnsw_dist_method": "vector_cosine_ops",
    },
)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [13]:
index = VectorStoreIndex(nodes, storage_context=storage_context, 
                             show_progress=True, embed_model=Settings.embed_model,
                            # transformations=[node_parser]
                        )

Generating embeddings:   0%|          | 0/1041 [00:00<?, ?it/s]

In [14]:
def completion_to_prompt(completion):
   return f"<|im_start|>system\n<|im_end|>\n<|im_start|>user\n{completion}<|im_end|>\n<|im_start|>assistant\n"

def messages_to_prompt(messages):
    prompt = ""
    for message in messages:
        if message.role == "system":
            prompt += f"<|im_start|>system\n{message.content}<|im_end|>\n"
        elif message.role == "user":
            prompt += f"<|im_start|>user\n{message.content}<|im_end|>\n"
        elif message.role == "assistant":
            prompt += f"<|im_start|>assistant\n{message.content}<|im_end|>\n"

    if not prompt.startswith("<|im_start|>system"):
        prompt = "<|im_start|>system\n" + prompt
        

    prompt = prompt + '<|im_start|>"You are Qwen, created by Alibaba Cloud. You are a helpful assistant. You answer user queries about the Thief Fan Mission forum on the Through The Looking Glass (TTLG) forums using retreived html data scraped from the boards. The html data contains the html body tag of each scraped page.\n'

    return prompt

llm = LlamaCPP(
    model_url="https://huggingface.co/bartowski/Qwen2.5-14B-Instruct-GGUF/resolve/main/Qwen2.5-14B-Instruct-Q6_K.gguf",
    temperature=0.1,
    max_new_tokens=1024,
    context_window=12384,
    generate_kwargs={"repeat_penalty": 1.15, "top_k": 0, "top_p": 0.5, "min_p": 0.1},
    model_kwargs={
        "n_gpu_layers": -1,
    },
    messages_to_prompt=messages_to_prompt,
    completion_to_prompt=completion_to_prompt,
    verbose=True,
)

Settings.llm = llm

ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    yes
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:
  Device 0: NVIDIA GeForce RTX 4060 Ti, compute capability 8.9, VMM: yes
llama_load_model_from_file: using device CUDA0 (NVIDIA GeForce RTX 4060 Ti) - 14465 MiB free
llama_model_loader: loaded meta data with 38 key-value pairs and 579 tensors from /llamaindex_cache/models/Qwen2.5-14B-Instruct-Q6_K.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen2.5 14B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:       

In [15]:
print(
    index.as_query_engine().query(
        'Using the provided context, answer the following query: What are some highly regarded fan missions for thief 2?'
    )
)

llama_perf_context_print:        load time =    9749.81 ms
llama_perf_context_print: prompt eval time =       0.00 ms /  7562 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =       0.00 ms /   100 runs   (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:       total time =   16893.01 ms /  7662 tokens


The provided context does not specify any particular highly regarded fan missions for Thief 2. However, it mentions that some of the sites listed in "Where to Download FMs" section have rankings/ratings for many of the FMs and The Circle has a review system where everyone can give their opinions on the missions. Therefore, you may want to visit these websites or forums (such as TTLG's Thief 2 fan mission discussion forum) to find out which fan missions are highly regarded by players.
